# Project 2: Language Market Index (LMI)
## Which programming languages should a technology consultancy be hiring and training for?

**Starting point — Project 1:** We began by analysing Stack Overflow post volume as a proxy for developer activity (see [Project 1: Is Stack Overflow Dying?](proj_1_so_decline_analysis.ipynb)). That analysis showed a sharp, sustained decline in posts across all languages from November 2022 onwards — the month ChatGPT launched. The data wasn't broken; the platform itself was declining as developers moved their questions to AI tools.

**The problem:** Any analysis using SO post volume as a primary signal after 2022 would classify every language as "declining" — not because languages were losing relevance, but because the measurement instrument was degrading. The signal was platform noise, not language signal.

**The solution — the LMI:** We replaced SO post volume with a composite index built from four independent data sources, each measuring a different dimension of a language's market position.

| Source | What it measures | Weight |
|--------|-----------------|--------|
| Adzuna job postings | What the market is paying for | 35% |
| GitHub Octoverse | What developers are actually building | 30% |
| Stack Overflow Developer Survey | What developers say they use | 25% |
| TIOBE Index | Industry recognition | 10% |

Each source is normalised to 0–100 using min-max scaling before the weighted composite is computed. The weighting model is explicit and adjustable — which makes the methodology defensible and stress-testable.

## 1. Setup & Pipeline

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('../pipeline'))
load_dotenv(os.path.join('..', '.env'))

from processing.normalize import run as run_normalize
from processing.score import compute_index, DEFAULT_WEIGHTS

PROCESSED_DIR = os.path.join('..', 'data', 'processed')
INDEX_PATH    = os.path.join(PROCESSED_DIR, 'index.csv')
NORM_PATH     = os.path.join(PROCESSED_DIR, 'normalized.csv')

print('Run pipeline/run.py to fetch fresh data before running this notebook.')
print(f'Index file exists: {os.path.exists(INDEX_PATH)}')

os.makedirs('../plots/lmi', exist_ok=True)


## 2. Load the Index

In [ ]:
index = pd.read_csv(INDEX_PATH)
norm  = pd.read_csv(NORM_PATH)

print(f'Index computed: {index["date"].iloc[0]}')
print(f'Languages ranked: {len(index)}')
print()
print(index[['rank', 'language', 'composite_score', 'lifecycle']].to_string(index=False))

## 3. Ranked Bar Chart

In [ ]:
LIFECYCLE_COLORS = {
    'Dominant':  '#5bc0f8',
    'Rising':    '#4ecb71',
    'Mature':    '#f0c040',
    'Declining': '#f07070',
    'Niche':     '#aaaaaa',
}

fig, ax = plt.subplots(figsize=(13, 8))
colors  = [LIFECYCLE_COLORS.get(lc, '#fff') for lc in index['lifecycle']]

bars = ax.barh(index['language'][::-1], index['composite_score'][::-1],
               color=colors[::-1], edgecolor='none')

for bar, score in zip(bars, index['composite_score'][::-1]):
    ax.text(score + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{score:.1f}', va='center', fontsize=10, color='white')

for label, color in LIFECYCLE_COLORS.items():
    ax.barh([], [], color=color, label=label)
ax.legend(title='Lifecycle', fontsize=10, title_fontsize=10, loc='lower right')

ax.set_title('Language Market Index (LMI) — Composite Score', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Composite Score (0–100)', fontsize=12)
ax.set_xlim(0, 110)
ax.set_facecolor('#0f1117')
fig.patch.set_facecolor('#0f1117')
ax.tick_params(colors='#c0c0c0', labelsize=11)
ax.xaxis.label.set_color('#c0c0c0')
ax.title.set_color('white')
for spine in ax.spines.values(): spine.set_edgecolor('#2d3148')
plt.tight_layout()
plt.savefig(os.path.join('..', 'plots', 'lmi', 'composite_scores.png'), dpi=150,
            facecolor=fig.get_facecolor())
plt.show()

## 4. Score Breakdown by Source

The composite score is the weighted sum of four normalised signals. This chart shows the contribution of each source per language — making the methodology transparent and auditable.

In [ ]:
score_cols = [c for c in index.columns if c.startswith('score_')]
SOURCE_LABELS = {
    'score_adzuna_total':     'Job Postings (35%)',
    'score_github_octoverse': 'GitHub Octoverse (30%)',
    'score_so_survey_usage':  'Developer Survey (25%)',
    'score_tiobe_rating':     'TIOBE Index (10%)',
}
source_colors = ['#5c6cfa', '#4ecb71', '#f0c040', '#f07070']

fig, ax = plt.subplots(figsize=(14, 9))
langs   = index['language'].values[::-1]
bottom  = np.zeros(len(langs))

for col, color in zip(score_cols, source_colors):
    vals = index[col].values[::-1]
    ax.barh(langs, vals, left=bottom, color=color, label=SOURCE_LABELS.get(col, col))
    bottom += vals

ax.set_title('Language Market Index (LMI) — Score Breakdown by Source',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Weighted Score Contribution', fontsize=12)
ax.legend(fontsize=10, loc='lower right')
ax.set_facecolor('#0f1117')
fig.patch.set_facecolor('#0f1117')
ax.tick_params(colors='#c0c0c0', labelsize=11)
ax.xaxis.label.set_color('#c0c0c0')
ax.title.set_color('white')
for spine in ax.spines.values(): spine.set_edgecolor('#2d3148')
plt.tight_layout()
plt.savefig(os.path.join('..', 'plots', 'lmi', 'score_breakdown_by_source.png'), dpi=150,
            facecolor=fig.get_facecolor())
plt.show()

## 5. Sensitivity Analysis — Does the Weighting Model Matter?

A composite index is only as good as its weighting assumptions. This section stress-tests the model by computing the index under three alternative weight configurations and checking whether the ranking is stable.

In [ ]:
SCENARIOS = {
    'Default (35/30/25/10)': DEFAULT_WEIGHTS,
    'Jobs-heavy (60/20/15/5)': {
        'adzuna_total': 0.60, 'github_octoverse': 0.20,
        'so_survey_usage': 0.15, 'tiobe_rating': 0.05
    },
    'GitHub-heavy (20/60/15/5)': {
        'adzuna_total': 0.20, 'github_octoverse': 0.60,
        'so_survey_usage': 0.15, 'tiobe_rating': 0.05
    },
    'Equal weights (25/25/25/25)': {
        'adzuna_total': 0.25, 'github_octoverse': 0.25,
        'so_survey_usage': 0.25, 'tiobe_rating': 0.25
    },
}

scenario_results = {}
for label, weights in SCENARIOS.items():
    try:
        result = compute_index(weights)
        scenario_results[label] = result.set_index('language')['composite_score']
        print(f'{label}: top 5 = {list(result["language"].head(5))}')
    except Exception as e:
        print(f'{label}: ERROR — {e}')

if len(scenario_results) > 1:
    comparison = pd.DataFrame(scenario_results)
    print('\nRanking stability across weighting scenarios:')
    print(comparison.round(1).to_string())

## 6. Strategic Recommendation

## Strategic Recommendation

*Composite index from four sources: Adzuna job postings (35%), GitHub Octoverse (30%), Stack Overflow Developer Survey (25%), TIOBE Index (10%). All sources normalised 0–100 before weighting. Last computed: May 2026.*

---

**Priority hire** — **Python**, **JavaScript**, **Java**, **TypeScript**

Dominant composite scores across all four signals. Represents the highest-confidence workforce investment. Python leads at 88.4/100 — its score is consistent across every weighting scenario tested.

**Maintain** — **C++**, **C#**, **PHP**, **Go**

Stable scores — strong existing base, not expanding rapidly. Preserve capability, do not scale headcount.

**Deprioritise** — **Rust**, **Kotlin**, **Swift**, **Ruby**, **Assembly**, **Perl**, **R**, **Scala**

Niche or declining scores across multiple signals. Reduce new hiring and L&D investment. Note: Rust and Kotlin show strong developer interest in surveys but weak job market signal — worth monitoring as leading indicators.

---

**Highest confidence: Python — 88.4 / 100**

Ranks first across every weighting scenario tested (jobs-heavy, GitHub-heavy, equal weights). Growth is structural — driven by AI/ML adoption — not platform-dependent. Any consultancy without a strong Python bench is already behind.

**Methodology note:** This index was built specifically to replace Stack Overflow post volume as a primary signal after that metric became unreliable post-November 2022. Using four independent sources makes the composite more robust to any single source becoming noisy or unrepresentative.